In [20]:
import cv2
import numpy as np
import os

video_path = "video.mp4"

os.makedirs("results/original", exist_ok=True)
os.makedirs("results/enhanced", exist_ok=True)
os.makedirs("results/mask", exist_ok=True)
os.makedirs("results/final", exist_ok=True)

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: can't open videp")
    exit()

frame_no = 0
save_no = 0
DAMAGE_COLOR = (0, 0, 255)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_no += 1

    if frame_no % 15 != 0:
        continue

    frame = cv2.resize(frame, (800, 500))
    original = frame.copy()

    # Road ROI - side grass/edges avoid
    roi_mask = np.zeros((500, 800), dtype=np.uint8)
    road_points = np.array([[
        (170, 230),
        (650, 230),
        (730, 500),
        (80, 500)
    ]], dtype=np.int32)
    cv2.fillPoly(roi_mask, road_points, 255)

    # Enhancement
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    blur = cv2.GaussianBlur(enhanced, (7, 7), 0)

    # Segmentation
    thresh = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        61,
        15
    )

    kernel = np.ones((3, 3), np.uint8)

    morph = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
    morph = cv2.morphologyEx(morph, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Remove grass / green region
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    green_mask = cv2.inRange(
        hsv,
        np.array([35, 40, 40]),
        np.array([85, 255, 255])
    )

    morph[green_mask > 0] = 0

    # Keep only road region
    morph = cv2.bitwise_and(morph, morph, mask=roi_mask)

    contours, _ = cv2.findContours(
        morph,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    final = frame.copy()

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area < 1200 or area > 7000:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        # Avoid side detections
        if x < 60 or x + w > 740:
            continue

        # Avoid very large boxes
        if w > 220 or h > 180:
            continue

        perimeter = cv2.arcLength(cnt, True)
        aspect_ratio = w / float(h)

        label = None

        # Crack = long thin
        if aspect_ratio > 2.8 and area < 2500:
            label = "CRACK"

        # Pothole = blob/round shape
        elif 2500 <= area <= 7000 and 0.5 <= aspect_ratio <= 2.0:
            label = "POTHOLE"

        # Alligator = medium irregular region
        elif 1200 <= area < 3500 and 0.5 <= aspect_ratio <= 3.0:
            label = "ALLIGATOR"

        if label is not None:
            cv2.drawContours(final, [cnt], -1, DAMAGE_COLOR, 2)
            cv2.rectangle(final, (x, y), (x + w, y + h), DAMAGE_COLOR, 1)

            cv2.putText(
                final,
                label,
                (x, y - 5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                DAMAGE_COLOR,
                1
            )

    cv2.imwrite(f"results/original/original_{save_no}.jpg", original)
    cv2.imwrite(f"results/enhanced/enhanced_{save_no}.jpg", enhanced)
    cv2.imwrite(f"results/mask/mask_{save_no}.jpg", morph)
    cv2.imwrite(f"results/final/final_{save_no}.jpg", final)

    save_no += 1

    cv2.imshow("Road Damage Detection", final)

    if cv2.waitKey(500) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

print("Segmentation Completed Successfully!")
print("Total saved frames:", save_no)

Segmentation Completed Successfully!
Total saved frames: 3
